# Phase 4: XGBoost Challenger Model & SHAP analysis
This notebook trains an XGBoost classifier as a challenger model. We tune hyperparameters using grid search, compare its raw performance against the scorecard, and execute SHAP analysis to extract risk drivers.


In [ ]:
import pandas as pd
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning
from xgboost_model import XGBoostChallenger


## 1. Load Data Splits
Load raw splits since trees handle missing values directly.


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
], ['home_ownership', 'purpose'])


## 2. Train XGBoost Model
Configure categorical label encodings and fit the tuned tree models.


In [ ]:
xgb_cols = woe_model.selected_features
X_train = train_df[xgb_cols].copy()
X_oot = oot_df[xgb_cols].copy()

# Ordinal encoding for categories
for col in ['home_ownership', 'purpose']:
    if col in xgb_cols:
        X_train[col] = X_train[col].astype('category').cat.codes
        X_oot[col] = X_oot[col].astype('category').cat.codes

challenger = XGBoostChallenger(target_col='target')
challenger.fit_and_tune(X_train, train_df['target'], X_oot, oot_df['target'])


## 3. SHAP Analysis
Generate local and global SHAP summary explanations to identify top risk drivers.


In [ ]:
import matplotlib.pyplot as plt
# Generate shap files
challenger.run_shap_analysis(X_train, 'outputs/reports/shap_analysis.html', 'outputs/reports/shap_summary.png')
print('SHAP charts saved. High-risk drivers can be inspected in output files.')


## 4. Feature Importance Rankings
Display standard XGBoost feature gain importances.


In [ ]:
imp_df = challenger.get_feature_importance()
print(imp_df)
